In [1]:
# ================================================
# OTT 이탈 예측 프로젝트 EDA
# ================================================
# 경로 설정
from pathlib import Path

DATA_SUBDIR = Path("_data") / "02_interim" / "260430_membership_v1(이상치, 이름변경)"
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / DATA_SUBDIR / "Membership_v1.csv").exists()
)
DATA_DIR = PROJECT_ROOT / DATA_SUBDIR

PATH_MEMBERSHIP = DATA_DIR / "Membership_v1.csv"
PATH_USER_MAP = DATA_DIR / "User_Mapping_v1.csv"
PATH_VIEW_HISTORY = DATA_DIR / "View_History_v1.csv"
OUTPUT_DIR = Path.cwd()   # 그래프 저장 경로
# ================================================

In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import ttest_ind, chi2_contingency
import os

sns.set_style("whitegrid")

FONT_PATH = Path(r"C:\Windows\Fonts\malgun.ttf")
if FONT_PATH.exists():
    fm.fontManager.addfont(FONT_PATH)

plt.rcParams.update({
    "font.family": "Malgun Gothic",
    "font.sans-serif": ["Malgun Gothic", "DejaVu Sans"],
    "axes.unicode_minus": False,
})
os.makedirs(OUTPUT_DIR, exist_ok=True)



In [3]:

# ────────────────────────────────────────────────
# 1. 데이터 로드 및 전처리
# ────────────────────────────────────────────────
mem = pd.read_csv(PATH_MEMBERSHIP)
um  = pd.read_csv(PATH_USER_MAP)
vh  = pd.read_csv(PATH_VIEW_HISTORY)

mem['reg_date'] = pd.to_datetime(mem['reg_date'])
mem['end_date'] = pd.to_datetime(mem['end_date'])
mem['membership_days'] = (mem['end_date'] - mem['reg_date']).dt.days

vh['watch_day'] = pd.to_datetime(vh['watch_day'], format='%Y%m%d')

# gender=N, is_user_verified=0, age=40 → 더미 이상치 제거
mask_dummy = (mem['gender'] == 'N') & (mem['is_user_verified'] == 0) & (mem['age'] == 40)
print(f"더미 이상치 제거: {mask_dummy.sum()}행 제거 ({mask_dummy.mean()*100:.1f}%)")
mem_clean = mem[~mask_dummy].copy()



더미 이상치 제거: 2639행 제거 (14.8%)


In [4]:

# ────────────────────────────────────────────────
# 2. View History 파생변수 생성
# ────────────────────────────────────────────────
START_DATE = pd.to_datetime('20210301', format='%Y%m%d')
vh['week'] = ((vh['watch_day'] - START_DATE).dt.days // 7 + 1).clip(1, 3)

# 주차별 시청시간
week_agg = (vh.groupby(['USER_NUM', 'week'])['watch_time(min)']
              .sum().unstack(fill_value=0))
week_agg.columns = [f'week{int(c)}_watch_time' for c in week_agg.columns]
week_agg = week_agg.reindex(columns=['week1_watch_time','week2_watch_time','week3_watch_time'], fill_value=0)

# 추이
week_agg['w1_to_w2'] = week_agg['week2_watch_time'] - week_agg['week1_watch_time']
week_agg['w2_to_w3'] = week_agg['week3_watch_time'] - week_agg['week2_watch_time']
week_agg['w1_to_w3'] = week_agg['week3_watch_time'] - week_agg['week1_watch_time']
week_agg['trend'] = week_agg['w1_to_w3'].apply(
    lambda x: 'increase' if x > 0 else ('decrease' if x < 0 else 'flat'))

# 기타 집계
vh_agg = vh.groupby('USER_NUM').agg(
    total_watch_time=('watch_time(min)', 'sum'),
    total_sessions=('watch_time(min)', 'count'),
    unique_contents=('MOVIE_NUM', 'nunique'),
    unique_days=('watch_day', 'nunique'),
    avg_session_time=('watch_time(min)', 'mean'),
    first_watch=('watch_day', 'min'),
    last_watch=('watch_day', 'max'),
).reset_index()

week_agg = week_agg.reset_index()
vh_all = vh_agg.merge(week_agg, on='USER_NUM', how='left')

# Membership + Mapping + ViewHistory 병합
merged = mem_clean.merge(um, on='USER_KEY', how='left')
merged = merged.merge(vh_all, on='USER_NUM', how='left')

# 마지막 시청 ~ 종료일 간격
merged['days_to_end'] = (merged['end_date'] - merged['last_watch']).dt.days
merged['days_to_first_watch'] = (merged['first_watch'] - merged['reg_date']).dt.days

# 시청이력 유무
merged['has_view'] = merged['total_watch_time'].notna().astype(int)
# 시청이력 없는 유저는 시청 관련 변수 0 처리
view_cols = ['total_watch_time','total_sessions','unique_contents','unique_days',
             'avg_session_time','week1_watch_time','week2_watch_time','week3_watch_time',
             'w1_to_w2','w2_to_w3','w1_to_w3']
merged[view_cols] = merged[view_cols].fillna(0)

print(f"\n최종 데이터셋: {len(merged)}명")
print(f"  시청이력 있는 유저: {merged['has_view'].sum()}명")
print(f"  재구독O: {merged['is_repurchase'].sum()}명 ({merged['is_repurchase'].mean()*100:.1f}%)")
print(f"  프로모션O: {(merged['is_promotion']==1).sum()}명 ({(merged['is_promotion']==1).mean()*100:.1f}%)")




최종 데이터셋: 15312명
  시청이력 있는 유저: 12775명
  재구독O: 10093명 (65.9%)
  프로모션O: 9069명 (59.2%)


In [5]:

# ────────────────────────────────────────────────
# 3. 유틸 함수
# ────────────────────────────────────────────────
def stat_test_cat(df, col, target='is_repurchase'):
    ct = pd.crosstab(df[col], df[target])
    chi2, p, _, _ = chi2_contingency(ct)
    rates = df.groupby(col)[target].mean().round(3)
    counts = df[col].value_counts()
    result = pd.DataFrame({'재구독률': rates, 'N': counts})
    sig = '★' if p < 0.05 else ''
    print(f"\n[{col}] chi2={chi2:.2f}, p={p:.4f} {sig}")
    print(result.to_string())

def stat_test_num(df, col, target='is_repurchase'):
    g1 = df[df[target]==1][col].dropna()
    g0 = df[df[target]==0][col].dropna()
    t, p = ttest_ind(g1, g0)
    sig = '★' if p < 0.05 else ''
    print(f"\n[{col}] t={t:.2f}, p={p:.4f} {sig}")
    print(f"  재구독O: {g1.mean():.2f} | 재구독X: {g0.mean():.2f} | 차이: {g1.mean()-g0.mean():.2f}")


In [6]:


# ────────────────────────────────────────────────
# 4. Membership 변수별 재구독률 분석
# ────────────────────────────────────────────────
print("\n" + "="*50)
print("4. Membership 변수 vs is_repurchase")
print("="*50)

cat_vars = ['is_promotion', 'max_screen', 'payment_device', 'is_churn_prevented',
            'gender', 'is_user_verified', 'billing_method']
num_vars = ['age', 'price', 'reg_hour', 'membership_days']

for col in cat_vars:
    stat_test_cat(merged, col)

for col in num_vars:
    stat_test_num(merged, col)




4. Membership 변수 vs is_repurchase

[is_promotion] chi2=109.34, p=0.0000 ★
               재구독률     N
is_promotion             
0             0.708  6243
1             0.626  9069

[max_screen] chi2=286.76, p=0.0000 ★
             재구독률     N
max_screen             
1           0.664  9477
2           0.746  3212
4           0.536  2623

[payment_device] chi2=178.71, p=0.0000 ★
                 재구독률     N
payment_device             
android         0.697  7803
ios             0.624  1412
mobile          0.590  2570
ott             0.863    51
pc              0.623  3273
smarttv         0.867   203

[is_churn_prevented] chi2=48.29, p=0.0000 ★
                     재구독률      N
is_churn_prevented              
0                   0.646  12266
1                   0.713   3046

[gender] chi2=7.33, p=0.0256 ★
         재구독률     N
gender             
F       0.653  9488
M       0.667  5707
N       0.752   117

[is_user_verified] chi2=7.93, p=0.0048 ★
                   재구독률      N
is_user_verifie

In [7]:

# ────────────────────────────────────────────────
# 5. View History 파생변수 vs 재구독률
# ────────────────────────────────────────────────
print("\n" + "="*50)
print("5. View History 파생변수 vs is_repurchase (시청이력 있는 유저)")
print("="*50)

df_v = merged[merged['has_view']==1].copy()
view_test_vars = ['total_watch_time','total_sessions','unique_contents','unique_days',
                  'avg_session_time','week1_watch_time','week2_watch_time','week3_watch_time',
                  'w1_to_w2','w2_to_w3','w1_to_w3','days_to_end','days_to_first_watch']

for col in view_test_vars:
    stat_test_num(df_v, col)



5. View History 파생변수 vs is_repurchase (시청이력 있는 유저)

[total_watch_time] t=-0.49, p=0.6267 
  재구독O: 318.18 | 재구독X: 321.17 | 차이: -2.98

[total_sessions] t=-1.20, p=0.2293 
  재구독O: 7.10 | 재구독X: 7.24 | 차이: -0.13

[unique_contents] t=-0.87, p=0.3850 
  재구독O: 5.09 | 재구독X: 5.16 | 차이: -0.07

[unique_days] t=-0.92, p=0.3573 
  재구독O: 3.90 | 재구독X: 3.95 | 차이: -0.04

[avg_session_time] t=1.00, p=0.3167 
  재구독O: 45.26 | 재구독X: 44.69 | 차이: 0.57

[week1_watch_time] t=2.10, p=0.0355 ★
  재구독O: 38.94 | 재구독X: 34.95 | 차이: 3.99

[week2_watch_time] t=1.37, p=0.1704 
  재구독O: 80.53 | 재구독X: 77.03 | 차이: 3.50

[week3_watch_time] t=-2.22, p=0.0262 ★
  재구독O: 198.71 | 재구독X: 209.19 | 차이: -10.47

[w1_to_w2] t=-0.18, p=0.8605 
  재구독O: 41.59 | 재구독X: 42.08 | 차이: -0.48

[w2_to_w3] t=-2.78, p=0.0055 ★
  재구독O: 118.18 | 재구독X: 132.16 | 차이: -13.98

[w1_to_w3] t=-2.82, p=0.0049 ★
  재구독O: 159.78 | 재구독X: 174.24 | 차이: -14.46

[days_to_end] t=17.81, p=0.0000 ★
  재구독O: 15.65 | 재구독X: 13.30 | 차이: 2.35

[days_to_first_watch] t=-1.73, p=

In [8]:


# ────────────────────────────────────────────────
# 6. 프로모션 O vs X 세부 비교
# ────────────────────────────────────────────────
print("\n" + "="*50)
print("6. 프로모션O vs 프로모션X 세부 비교")
print("="*50)

p1 = merged[merged['is_promotion']==1]
p0 = merged[merged['is_promotion']==0]
print(f"\n프로모션O 재구독률: {p1['is_repurchase'].mean()*100:.2f}% (N={len(p1)})")
print(f"프로모션X 재구독률: {p0['is_repurchase'].mean()*100:.2f}% (N={len(p0)})")

# 연령대별
merged['age_group'] = pd.cut(merged['age'], bins=[0,20,30,40,50,100],
                              labels=['~20','21~30','31~40','41~50','51~'])
print("\n연령대 × 프로모션 재구독률:")
print(merged.groupby(['age_group','is_promotion'], observed=True)['is_repurchase']
      .mean().unstack().round(3))

# max_screen별
print("\nmax_screen × 프로모션 재구독률:")
print(merged.groupby(['max_screen','is_promotion'])['is_repurchase']
      .mean().unstack().round(3))

# 주차별 시청시간
print("\n주차별 시청시간 (프로모션 비교):")
for col in ['week1_watch_time','week2_watch_time','week3_watch_time']:
    t, p = ttest_ind(p1[col], p0[col])
    sig = '★' if p < 0.05 else ''
    print(f"  [{col}] O={p1[col].mean():.1f} | X={p0[col].mean():.1f} | p={p:.4f} {sig}")

# 마지막 시청 ~ 종료일 간격
merged['days_to_end_group'] = pd.cut(merged['days_to_end'],
    bins=[-1,0,3,7,14,100], labels=['당일','1~3일','4~7일','8~14일','15일+'])
print("\n마지막 시청~종료일 간격 × 프로모션 재구독률:")
print(merged.groupby(['days_to_end_group','is_promotion'], observed=True)['is_repurchase']
      .mean().unstack().round(3))




6. 프로모션O vs 프로모션X 세부 비교

프로모션O 재구독률: 62.59% (N=9069)
프로모션X 재구독률: 70.75% (N=6243)

연령대 × 프로모션 재구독률:
is_promotion      0      1
age_group                 
~20           0.607  0.499
21~30         0.724  0.649
31~40         0.709  0.683
41~50         0.766  0.660
51~           0.736  0.659

max_screen × 프로모션 재구독률:
is_promotion      0      1
max_screen                
1             0.685  0.649
2             0.753  0.738
4             0.731  0.480

주차별 시청시간 (프로모션 비교):
  [week1_watch_time] O=29.4 | X=34.2 | p=0.0020 ★
  [week2_watch_time] O=67.3 | X=64.6 | p=0.2047 
  [week3_watch_time] O=171.5 | X=164.8 | p=0.0953 

마지막 시청~종료일 간격 × 프로모션 재구독률:
is_promotion           0      1
days_to_end_group              
당일                 0.000  0.000
1~3일               0.000  0.000
4~7일               0.000  0.000
8~14일              0.725  0.628
15일+               0.753  0.634


In [9]:

# ────────────────────────────────────────────────
# 7. 시각화
# ────────────────────────────────────────────────
print("\n시각화 생성 중...")

# (1) 재구독률 전체 파이
fig, ax = plt.subplots(figsize=(5,5))
vals = merged['is_repurchase'].value_counts()
ax.pie(vals, labels=['재구독O','재구독X'], autopct='%1.1f%%',
       colors=['#378ADD','#D4537E'], startangle=90)
ax.set_title('전체 재구독률')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_repurchase_overall.png'), dpi=150)
plt.close()

# (2) 프로모션 O/X 재구독률 비교 막대
fig, ax = plt.subplots(figsize=(6,4))
promo_rates = merged.groupby('is_promotion')['is_repurchase'].mean() * 100
bars = ax.bar(['프로모션X','프로모션O'], promo_rates[[0,1]],
              color=['#1D9E75','#378ADD'], width=0.5)
for bar, val in zip(bars, promo_rates[[0,1]]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 85)
ax.set_ylabel('재구독률 (%)')
ax.set_title('프로모션 여부별 재구독률')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_repurchase_by_promotion.png'), dpi=150)
plt.close()

# (3) max_screen별 재구독률
fig, ax = plt.subplots(figsize=(6,4))
screen_rates = merged.groupby('max_screen')['is_repurchase'].mean() * 100
bars = ax.bar([f'{s}인 요금제' for s in screen_rates.index], screen_rates.values,
              color=['#378ADD','#1D9E75','#D4537E'], width=0.5)
for bar, val in zip(bars, screen_rates.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 85)
ax.set_ylabel('재구독률 (%)')
ax.set_title('요금제(max_screen)별 재구독률')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_repurchase_by_maxscreen.png'), dpi=150)
plt.close()

# (4) 연령대 × 프로모션 재구독률 히트맵
pivot = merged.groupby(['age_group','is_promotion'], observed=True)['is_repurchase'].mean().unstack()
pivot.columns = ['프로모션X','프로모션O']
fig, ax = plt.subplots(figsize=(6,4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            linewidths=0.5, vmin=0.4, vmax=0.85)
ax.set_title('연령대 × 프로모션 재구독률')
ax.set_xlabel('')
ax.set_ylabel('연령대')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_heatmap_age_promotion.png'), dpi=150)
plt.close()

# (5) 주차별 시청시간 추이 (프로모션 O/X)
fig, ax = plt.subplots(figsize=(7,4))
weeks = ['1주차','2주차','3주차']
cols = ['week1_watch_time','week2_watch_time','week3_watch_time']
ax.plot(weeks, [p1[c].mean() for c in cols], marker='o', label='프로모션O',
        color='#378ADD', linewidth=2)
ax.plot(weeks, [p0[c].mean() for c in cols], marker='o', label='프로모션X',
        color='#1D9E75', linewidth=2)
ax.set_ylabel('평균 시청시간 (분)')
ax.set_title('주차별 평균 시청시간 추이 (프로모션 구분)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_weekly_watch_by_promotion.png'), dpi=150)
plt.close()

# (6) 주차별 시청시간 추이 (재구독 O/X)
fig, ax = plt.subplots(figsize=(7,4))
r1 = merged[merged['is_repurchase']==1]
r0 = merged[merged['is_repurchase']==0]
ax.plot(weeks, [r1[c].mean() for c in cols], marker='o', label='재구독O',
        color='#378ADD', linewidth=2)
ax.plot(weeks, [r0[c].mean() for c in cols], marker='o', label='재구독X',
        color='#D4537E', linewidth=2)
ax.set_ylabel('평균 시청시간 (분)')
ax.set_title('주차별 평균 시청시간 추이 (재구독 구분)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_weekly_watch_by_repurchase.png'), dpi=150)
plt.close()

# (7) days_to_end × 프로모션 재구독률
fig, ax = plt.subplots(figsize=(7,4))
dte = merged.groupby(['days_to_end_group','is_promotion'], observed=True)['is_repurchase'].mean().unstack() * 100
dte.columns = ['프로모션X','프로모션O']
dte[['프로모션X','프로모션O']].plot(kind='bar', ax=ax,
    color=['#1D9E75','#378ADD'], width=0.6)
ax.set_xlabel('마지막 시청 ~ 종료일 간격')
ax.set_ylabel('재구독률 (%)')
ax.set_title('마지막 시청~종료일 간격 × 프로모션 재구독률')
ax.legend()
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_days_to_end_promotion.png'), dpi=150)
plt.close()

# (8) 시청 추이 패턴 분포 (프로모션 구분)
fig, ax = plt.subplots(figsize=(6,4))
trend_dist = merged[merged['has_view']==1].groupby(
    ['is_promotion','trend'])['USER_KEY'].count().unstack(fill_value=0)
trend_dist = trend_dist.div(trend_dist.sum(axis=1), axis=0) * 100
trend_dist.index = ['프로모션X','프로모션O']
trend_dist[['increase','flat','decrease']].plot(kind='bar', ax=ax,
    color=['#378ADD','#aaaaaa','#D4537E'], width=0.5)
ax.set_ylabel('비율 (%)')
ax.set_title('시청 추이 패턴 분포 (프로모션 구분)')
ax.legend(['증가','유지','감소'])
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '08_trend_pattern_by_promotion.png'), dpi=150)
plt.close()

print("\n완료. 저장된 파일:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.png'):
        print(f"  {f}")


시각화 생성 중...

완료. 저장된 파일:
  01_repurchase_overall.png
  02_repurchase_by_promotion.png
  03_repurchase_by_maxscreen.png
  04_heatmap_age_promotion.png
  05_weekly_watch_by_promotion.png
  06_weekly_watch_by_repurchase.png
  07_days_to_end_promotion.png
  08_trend_pattern_by_promotion.png
